# **AI METHODS I**
## Hackathon · Notebook 2: Bring your model online

In notebook 1 you trained an autoencoder and saved it in `model_app/`. This notebook takes it all the way to a **live app**: a web page behind a password that sends an input to **your model** and stores every result in a **database**.

| Day | Goal | Sections |
|---|---|---|
| **Tuesday** | The model runs as a **model server** (Gradio), on your computer and online (Render), and another program can call it | 1, 2 |
| **Wednesday** | A **Cloudflare** web app: page, password, `/api/predict`, database (D1) and file storage (R2) | 3 |
| **Thursday** | Diagrams, failure cases, the final app | 4 |
| **Friday** | Live demo checklist | 5 |

Everything here is a **working example**, tested end to end:
- `model_app/` is the model server: the anomaly detector of notebook 1;
- `cloudflare_app/` is the web app.

Both folders sit next to this notebook. Copy them, then change the model, the page and the table to fit your project.

### Learning goals
By the end of the week you will be able to:
1. Serve a model as a small web app with a page for people and an **API** for programs, and put it online.
2. Call a model **over HTTP** from Python and from JavaScript.
3. Build a small **serverless** web app: static page + functions + database + file storage + password.
4. Draw the **architecture**, the **user flow** and the **sequence** of your app, and explain what happens when something fails.

### Table of contents
0. Setup
1. The architecture
2. Tuesday: the model server
3. Wednesday: the Cloudflare app
4. Thursday: diagrams and failure cases
5. Friday: demo checklist
6. Conclusions

# 0. Setup

💡 **Tip – Google Colab:** sections 1 and 2 work in Colab, except starting the model server locally (section 2.5), which needs your own computer. The Cloudflare part (section 3) runs in a **terminal** on your computer.

The model server needs `gradio` and `onnxruntime`. Both are in the course `requirements.txt`, together with `gradio_client`, used here to call the server.

In [1]:
# Only needed in Google Colab or a new environment: remove the "#" and run this cell once.
# %pip install gradio gradio_client onnxruntime requests pillow

In [2]:
import base64
import importlib.util
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import requests
from gradio_client import Client

MODEL_APP_DIR = Path("model_app")          # the model server (written by notebook 1, section 9)
APP_DIR = Path("cloudflare_app")           # the Cloudflare app

# Where the model server answers. Locally: http://127.0.0.1:7860
# Online, after section 2.6: "https://<your-service-name>.onrender.com"
MODEL_URL = "http://127.0.0.1:7860"


def show_file(path, first_line=None, last_line=None):
    # Print a file (or a part of it) with line numbers: the real file, so what you copy is what was tested
    lines = Path(path).read_text().splitlines()
    first, last = (first_line or 1), (last_line or len(lines))
    print(f"--- {path} (lines {first}-{last} of {len(lines)})")
    for n in range(first, last + 1):
        print(f"{n:4d}  {lines[n - 1]}")


def image_to_base64(path):
    # HTTP requests carry text: base64 turns a file into text
    return base64.b64encode(Path(path).read_bytes()).decode()

# 1. The architecture

Your app has four parts. Each does **one** job, and only the Function talks to the model:

```
 ┌───────────┐      ┌────────────────────── Cloudflare ──────────────────────┐      ┌──────────────────┐
 │  Browser  │      │  Pages: the page (public/)                              │      │ Model server     │
 │  (user)   │ ───▶ │  Middleware: password gate (functions/_middleware.js)   │      │ (Render)         │
 │           │ ◀─── │  Function /api/predict (functions/api/predict.js)  ─────┼────▶ │ your autoencoder │
 └───────────┘      │       │                     │                           │ ◀─── │ (app.py + .onnx) │
                    │       ▼                     ▼                           │      └──────────────────┘
                    │  D1 database           R2 bucket                        │
                    │  (one row per result)  (the uploaded files)             │
                    └─────────────────────────────────────────────────────────┘
```

| Part | Job | Why not somewhere else? |
|---|---|---|
| **Pages** | Serves the HTML page | A static page is free and fast, but it can't keep secrets: anyone can read its code |
| **Function** (`/api/...`) | Checks the password, calls the model, saves the result | Runs on Cloudflare's servers, so passwords and keys stay hidden |
| **Model server** | Runs the model | Python models don't run inside Cloudflare Functions |
| **D1** | Stores results as rows (SQL) | Data in the browser disappears on reload and is visible only to one person |
| **R2** | Stores files (images, audio) | Databases are for small values, not for files |

### The sequence: what happens when a user clicks "Analyse"

```
 Browser                 Cloudflare Function             Model server             D1        R2
    │  POST /api/predict        │                            │                      │         │
    │  (image + auth cookie)    │                            │                      │         │
    │ ─────────────────────────▶│ cookie OK?                 │                      │         │
    │                           │ POST /gradio_api/call/predict (base64 image)      │         │
    │                           │ ──────────────────────────▶│                      │         │
    │                           │ ◀──────────── event_id ────│                      │         │
    │                           │ GET .../predict/event_id   │                      │         │
    │                           │ ──────────────────────────▶│ run the autoencoder  │         │
    │                           │ ◀──── score, verdict ──────│                      │         │
    │                           │ PUT uploads/<id> ─────────────────────────────────┼────────▶│
    │                           │ INSERT INTO predictions ──────────────────────────▶│         │
    │ ◀──── JSON result ────────│                            │                      │         │
    │  show verdict             │                            │                      │         │
    │  GET /api/history ───────▶│ SELECT ... LIMIT 20 ──────────────────────────────▶│         │
```

💡 **Tip:** this is the kind of diagram you hand in on Wednesday and Friday. ASCII is fine; so is a drawing in Miro.

# 2. Tuesday: the model server

A **model server** is a small web app that loads your model once and answers requests: an input comes in, the prediction goes out. We build it with **Gradio**, a Python library that creates, from one file (`app.py`):
- a **page** for people (upload an image, see the result);
- an **API** for programs (send text, get JSON back).

The server runs on your computer first, then online on **Render**, which hosts small web services for free.

⚠️ **Watch out – why not Hugging Face Spaces?** Since mid-2026, Hugging Face asks for a paid plan to run Gradio Spaces, both for organisations and for personal accounts; only static Spaces stay free. Hugging Face is still the place to **share** models and datasets (the `iaac-maai` organisation), but the model **server** runs on Render.

## 2.1 The files

In [3]:
for path in sorted(MODEL_APP_DIR.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{str(path):40s} {path.stat().st_size / 1024:8.1f} KB")

model_app/.python-version                     0.0 KB
model_app/README.md                           3.3 KB
model_app/anomaly_ae.onnx                   259.2 KB
model_app/anomaly_ae.pt                     244.4 KB
model_app/anomaly_ae_config.json              0.5 KB
model_app/app.py                              4.3 KB
model_app/examples/bag.png                    1.6 KB
model_app/examples/coat.png                   1.4 KB
model_app/examples/sneaker.png                0.9 KB
model_app/examples/t-shirt.png                1.3 KB
model_app/render.yaml                         0.3 KB
model_app/requirements.txt                    0.1 KB


| File | What it does |
|---|---|
| `app.py` | Loads the model, prepares the input, defines the page **and** the API |
| `anomaly_ae.onnx` | The model: architecture and weights (notebook 1, section 9) |
| `anomaly_ae_config.json` | Image size, preprocessing, threshold |
| `examples/` | Test images for the page |
| `requirements.txt` | The Python packages the server installs: **no PyTorch**, ONNX Runtime runs the model |
| `.python-version`, `render.yaml` | Tell Render which Python to use and how to build and start the server |
| `README.md` | The **explanation of your autoencoder** (your Tuesday homework) and how to run it |

## 2.2 `app.py`, part by part

**Loading and preprocessing.** The ONNX file already contains the network, so there is no model class to copy. The **preprocessing** must be exactly the one used in training: if the server prepares images differently (another size, no scaling, colours inverted), it gives wrong answers **without any error message**.

In [4]:
show_file(MODEL_APP_DIR / "app.py", 1, 58)

--- model_app/app.py (lines 1-58 of 106)
   1  """Model server: anomaly detection with a convolutional autoencoder (Gradio + ONNX Runtime).
   2  
   3  The page has a form for people (upload an image, see the result) and an API
   4  for programs (`/predict`: send a base64 image, get JSON back). The Cloudflare
   5  app of notebook 2 uses the API. Runs locally (`python app.py`) and on Render.
   6  
   7  The model is the ONNX file exported in notebook 1: it contains the architecture
   8  AND the weights, so this file needs no PyTorch and no copy of the model class.
   9  """
  10  import base64
  11  import io
  12  import json
  13  import os
  14  from pathlib import Path
  15  
  16  import gradio as gr
  17  import numpy as np
  18  import onnxruntime as ort
  19  from PIL import Image
  20  
  21  HERE = Path(__file__).parent
  22  CONFIG = json.loads((HERE / "anomaly_ae_config.json").read_text())
  23  SIZE = CONFIG["image_size"]
  24  THRESHOLD = CONFIG["threshold"]
  25  
  

**The page and the API.** `analyse_for_page` feeds the web form (an image in, a verdict and two images out). `predict` is for **programs**: a base64 text in, a JSON dictionary out, which is easy to send from JavaScript. `gr.api(predict, api_name="predict")` publishes it at `/predict` without adding anything to the page. The last lines start the server on the port the host asks for (`PORT`), or 7860 on your computer.

In [5]:
show_file(MODEL_APP_DIR / "app.py", 60, None)

--- model_app/app.py (lines 60-106 of 106)
  60  # ---- For people: the web form ------------------------------------------------
  61  def analyse_for_page(image):
  62      if image is None:
  63          raise gr.Error("Upload an image first.")
  64      result, rebuilt, error_map = analyse(image)
  65      verdict = "Anomaly" if result["is_anomaly"] else "Normal"
  66      return {verdict: 1.0}, result, to_image(rebuilt), to_image(error_map / max(error_map.max(), 1e-8))
  67  
  68  
  69  # ---- For programs: base64 image in, JSON out ---------------------------------
  70  def predict(image_b64: str) -> dict:
  71      """Score one image. Send a base64-encoded PNG or JPEG (a 'data:image/...;base64,' prefix is allowed)."""
  72      if "," in image_b64[:100]:
  73          image_b64 = image_b64.split(",", 1)[1]
  74      image = Image.open(io.BytesIO(base64.b64decode(image_b64)))
  75      result, rebuilt, _ = analyse(image)
  76      buffer = io.BytesIO()
  77      to_image(rebui

💡 **Tip – base64:** HTTP requests carry text. Base64 turns any file (image, audio, CSV) into text and back. It makes files about a third bigger, so keep uploads small.

## 2.3 Test `predict` without a server

Before starting anything, load `app.py` as a module and call `predict` directly. A mistake here costs seconds; on Render, every attempt costs a rebuild of a few minutes.

In [6]:
spec = importlib.util.spec_from_file_location("model_app_module", MODEL_APP_DIR / "app.py")
model_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_app)          # runs app.py, but does not start the web server

for example in sorted((MODEL_APP_DIR / "examples").glob("*.png")):
    result = model_app.predict(image_to_base64(example))
    print(f"{example.name:12s} score {result['score']:.4f}  ({result['score_vs_threshold']}× threshold)  "
          f"anomaly={result['is_anomaly']}")

bag.png      score 0.0455  (1.3× threshold)  anomaly=True
coat.png     score 0.0124  (0.35× threshold)  anomaly=False
sneaker.png  score 0.0595  (1.7× threshold)  anomaly=True
t-shirt.png  score 0.0237  (0.68× threshold)  anomaly=False


## 2.4 Start the server on your computer

In a terminal, inside `model_app/`:

```bash
python app.py
```

Then open http://127.0.0.1:7860: that is the page for people. Stop it with `Ctrl+C`.

The next cell does the same from the notebook, in the background, so we can call the server from the following cells. It waits until the server answers.

In [7]:
def start_local_server(folder, port=7860, timeout=60):
    # Start `python app.py` in the background and wait until it answers
    url = f"http://127.0.0.1:{port}"
    try:
        requests.get(url, timeout=2)
        print(f"A server is already running at {url}")
        return None
    except requests.ConnectionError:
        pass
    process = subprocess.Popen([sys.executable, "app.py"], cwd=folder, env={**os.environ, "PORT": str(port)},
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    start = time.time()
    while time.time() - start < timeout:
        try:
            requests.get(url, timeout=2)
            print(f"Server running at {url} (started in {time.time() - start:.1f} s)")
            return process
        except requests.ConnectionError:
            time.sleep(0.5)
    process.terminate()
    raise RuntimeError("The server did not start: run `python app.py` in a terminal to see the error")


server = start_local_server(MODEL_APP_DIR) if MODEL_URL.startswith("http://127.0.0.1") else None

Server running at http://127.0.0.1:7860 (started in 2.0 s)


## 2.5 Call it from another program

This is Tuesday's goal: the model answers **another program**, not just a person on the page. First with `gradio_client`, the easy way from Python:

In [8]:
client = Client(MODEL_URL, verbose=False)
start = time.time()
answer = client.predict(image_to_base64(MODEL_APP_DIR / "examples" / "sneaker.png"), api_name="/predict")
print(f"answered in {time.time() - start:.2f} s")
print({k: v for k, v in answer.items() if k != "reconstruction_png_base64"})

answered in 0.12 s
{'score': 0.05946, 'threshold': 0.03506, 'is_anomaly': True, 'score_vs_threshold': 1.7, 'inverted_input': False}


Then with plain HTTP requests. These are **exactly** the two calls the Cloudflare app makes in JavaScript (section 3):
1. `POST /gradio_api/call/predict` with the input → the server answers with an `event_id` (a ticket number);
2. `GET /gradio_api/call/predict/<event_id>` → the result, after the line `event: complete`.

In [9]:
def call_model(url, image_b64, api_name="predict", timeout=90):
    # Gradio's HTTP API in two steps: start a job, then read its result
    ticket = requests.post(f"{url}/gradio_api/call/{api_name}", json={"data": [image_b64]}, timeout=timeout)
    ticket.raise_for_status()
    event_id = ticket.json()["event_id"]
    stream = requests.get(f"{url}/gradio_api/call/{api_name}/{event_id}", timeout=timeout).text
    if "event: complete" not in stream:
        raise RuntimeError(f"No result: {stream[:300]}")
    return json.loads(stream.split("event: complete")[1].split("data:")[1].strip())[0]


start = time.time()
result = call_model(MODEL_URL, image_to_base64(MODEL_APP_DIR / "examples" / "coat.png"))
print(f"answered in {time.time() - start:.2f} s:", {k: v for k, v in result.items() if k != "reconstruction_png_base64"})

answered in 0.01 s: {'score': 0.01244, 'threshold': 0.03506, 'is_anomaly': False, 'score_vs_threshold': 0.35, 'inverted_input': False}


Both calls get the same answer as the direct test in section 2.3: the sneaker scores **1.70×** the threshold (anomaly), the coat **0.35×** (normal). The server answers in a fraction of a second, because the model is tiny and already loaded. On Render the first call after a pause takes longer (section 2.7).

🤔 **Think about it:** `gradio_client` hides the two steps; `requests` shows them. When something breaks in the Cloudflare app on Wednesday, this cell is the quickest way to check whether the problem is the model server or the app.

## 2.6 Put it online with Render

[Render](https://render.com) runs small web services for free, straight from a GitHub repository. The free service has 512 MB of memory and **falls asleep after 15 minutes without requests**: the next request wakes it up, which takes about a minute.

1. **A repository for the server.** Put the contents of your `model_app/` folder (with your model, config, `requirements.txt`, `.python-version` and `render.yaml`) into its own GitHub repository in the `iaac-maai` organisation.
2. **Render account.** On [render.com](https://render.com), sign in with your GitHub account. Free web services don't need a credit card.
3. **Create the service.** Choose **New → Blueprint** and select the repository. Render reads `render.yaml`:

In [10]:
show_file(MODEL_APP_DIR / "render.yaml")
show_file(MODEL_APP_DIR / "requirements.txt")

--- model_app/render.yaml (lines 1-9 of 9)
   1  # Render "Blueprint": lets Render create the web service from this repository in one step
   2  services:
   3    - type: web
   4      name: hackathon-example-autoencoder
   5      runtime: python
   6      plan: free
   7      buildCommand: pip install -r requirements.txt
   8      startCommand: python app.py
   9      healthCheckPath: /
--- model_app/requirements.txt (lines 1-5 of 5)
   1  # The model server needs no PyTorch: the ONNX file runs with ONNX Runtime (small and fast on a CPU)
   2  gradio==6.32.0
   3  onnxruntime==1.31.1
   4  numpy
   5  pillow


4. **Wait for the build** (a few minutes; follow the "Logs" tab). When it says the service is live, it answers at `https://<service-name>.onrender.com`.
5. **Call it from here:** set `MODEL_URL` in the setup cell to that address, and run sections 2.5 and 2.7 again.

⚠️ **Watch out – memory:** 512 MB is enough for ONNX Runtime and Gradio (the example uses less than 200 MB), but **not** for PyTorch plus Gradio (about 420 MB, close to the limit). That is why notebook 1 exports the model to ONNX. If the log says the service ran **out of memory**, check that `torch` is not in `requirements.txt`.

⚠️ **Watch out – the README is your homework.** Explain your autoencoder in `README.md`: data, architecture, training, what the score means, the threshold and its limits. Use the example's `model_app/README.md` as a template.

💡 **Tip – sharing the model:** the `.onnx` file and the README can also go to a **model repository** in the `iaac-maai` organisation on Hugging Face (free). That is the place where others find and download your model; Render is where it runs.

## 2.7 Sleeping servers

A sleeping server answers the first request slowly or not at all. This helper calls the server until it answers, and reports how long that took:

In [11]:
def wake_up(url, tries=10, wait=10):
    for attempt in range(1, tries + 1):
        start = time.time()
        try:
            requests.get(url, timeout=60).raise_for_status()
            print(f"awake: answered in {time.time() - start:.1f} s (attempt {attempt})")
            return True
        except requests.RequestException as error:
            print(f"attempt {attempt}: not yet ({type(error).__name__}), waiting {wait} s")
            time.sleep(wait)
    return False


wake_up(MODEL_URL)

awake: answered in 0.0 s (attempt 1)


True

⚠️ **Watch out – Friday:** open your server's address **before 10:00** on Friday and run `wake_up`. A demo that starts by waking up a server starts with a minute of silence. During the demo, one request every few minutes keeps it awake.

### Your turn (Tuesday)
1. Copy `model_app/` into a new repository. Replace the ONNX model, the config, the examples and the **preprocessing** in `app.py` with yours. Keep the `predict` API: base64 (or numbers) in, a JSON dictionary out.
2. Test `predict` without a server (2.3), then start the server locally and call it (2.4, 2.5).
3. Put it on Render (2.6) and call the online address from this notebook with `gradio_client` **and** with `requests`.
4. Write the README: what your autoencoder does, how it was trained, what the output means.

💡 **Tip – inputs that are not images:** for a table, let `predict` take a JSON text with the column values (`json.loads(text)`); for a sensor window, a list of numbers; for audio, a base64 WAV. The two-step HTTP call stays the same.

# 3. Wednesday: the Cloudflare app

Cloudflare runs your app on its servers, on a free plan that is more than enough for the hackathon. We use **Cloudflare Pages** with **Functions**:
- everything in `public/` is the website (HTML, CSS, images);
- every file in `functions/` becomes a small server program for one address: `functions/api/predict.js` answers `/api/predict`;
- `functions/_middleware.js` runs **before every request**: our password gate.

Everything below runs in a **terminal**, inside the `cloudflare_app/` folder.

## 3.1 Tools and accounts

1. **Node.js LTS** from [nodejs.org](https://nodejs.org) (it includes `npm` and `npx`). Check with `node --version`.
2. In `cloudflare_app/`: `npm install` installs **Wrangler**, Cloudflare's command-line tool, at the version pinned in `package.json`.
3. A free **Cloudflare account**, then `npx wrangler login` (opens the browser once).

⚠️ **Watch out – Linux/WSL:** if `node` stops with `libatomic.so.1: cannot open shared object file`, run `sudo apt install libatomic1`. If `npm install` warns that install scripts were blocked, that is fine: Wrangler still works.

## 3.2 The project

In [12]:
skip = {"node_modules", ".wrangler"}
for path in sorted(APP_DIR.rglob("*")):
    if path.is_file() and not skip.intersection(path.parts) and path.name not in ("package-lock.json", ".dev.vars"):
        print(f"{str(path.relative_to(APP_DIR)):30s} {path.stat().st_size / 1024:6.1f} KB")

.gitignore                        0.0 KB
functions/_middleware.js          1.2 KB
functions/api/history.js          0.3 KB
functions/api/images/[id].js      0.4 KB
functions/api/login.js            0.9 KB
functions/api/predict.js          3.0 KB
package.json                      0.5 KB
public/index.html                 3.7 KB
public/login.html                 0.9 KB
public/style.css                  1.5 KB
schema.sql                        0.8 KB
wrangler.toml                     1.0 KB


| File | Job |
|---|---|
| `wrangler.toml` | The settings: project name, `MODEL_URL`, and the **bindings** that connect the functions to D1 (`env.DB`) and R2 (`env.BUCKET`) |
| `package.json` | The Wrangler version and short commands (`npm run dev`, `npm run deploy`, …) |
| `schema.sql` | The database table |
| `public/index.html` | The page: upload form, result, saved results |
| `public/login.html` | The login form |
| `functions/_middleware.js` | The password gate |
| `functions/api/login.js` | Checks the password and sets a cookie |
| `functions/api/predict.js` | Calls the model server, saves the image in R2 and the result in D1 |
| `functions/api/history.js` | Reads the last 20 results from D1 |
| `functions/api/images/[id].js` | Sends back a stored image (`[id]` is a part of the address, e.g. `/api/images/1234`) |

In [13]:
show_file(APP_DIR / "wrangler.toml")

--- cloudflare_app/wrangler.toml (lines 1-25 of 25)
   1  # Cloudflare Pages project: the page (public/) + the API (functions/) + D1 database + R2 bucket
   2  name = "hackathon-example-app"
   3  pages_build_output_dir = "./public"
   4  compatibility_date = "2026-11-01"
   5  
   6  # Plain settings, readable by the functions as env.MODEL_URL
   7  [vars]
   8  # Your model server (notebook 2, section 2): its Render address, without a slash at the end
   9  MODEL_URL = "https://hackathon-example-autoencoder.onrender.com"
  10  MODEL_API_NAME = "predict"
  11  
  12  # The database: `npx wrangler d1 create hackathon-example-db` prints the id to paste here
  13  [[d1_databases]]
  14  binding = "DB"
  15  database_name = "hackathon-example-db"
  16  database_id = "paste-the-id-from-wrangler-d1-create"
  17  
  18  # File storage for the uploaded images: `npx wrangler r2 bucket create hackathon-example-images`
  19  [[r2_buckets]]
  20  binding = "BUCKET"
  21  bucket_name = "hackathon-

## 3.3 The password gate

The page and the API must not be public. The middleware lets everyone reach the login page; everything else needs a cookie that proves the password was entered. The password itself is a **secret**: it never appears in the code or in git.

In [14]:
show_file(APP_DIR / "functions" / "_middleware.js")
show_file(APP_DIR / "functions" / "api" / "login.js")

--- cloudflare_app/functions/_middleware.js (lines 1-32 of 32)
   1  // Runs before EVERY request: the password gate.
   2  // A visitor without the right cookie only gets the login page and the login API.
   3  
   4  // Pages serves login.html at /login (it drops the .html)
   5  const PUBLIC_PATHS = ["/login", "/login.html", "/api/login", "/style.css"];
   6  
   7  export async function onRequest(context) {
   8    const { request, env, next } = context;
   9    const url = new URL(request.url);
  10  
  11    if (PUBLIC_PATHS.includes(url.pathname)) {
  12      return next();
  13    }
  14  
  15    const cookie = request.headers.get("Cookie") || "";
  16    const token = cookie.match(/(?:^|;\s*)auth=([a-f0-9]+)/)?.[1];
  17    if (token && token === (await authToken(env.APP_PASSWORD))) {
  18      return next(); // logged in: serve the page or run the API function
  19    }
  20  
  21    if (url.pathname.startsWith("/api/")) {
  22      return Response.json({ error: "Not logged

⚠️ **Watch out:** a password in the **page's** JavaScript (`if (input === "1234")`) protects nothing: anyone can open the page source. The check must happen in a function, on the server.

## 3.4 `/api/predict`: model, file, database

In [15]:
show_file(APP_DIR / "functions" / "api" / "predict.js")

--- cloudflare_app/functions/api/predict.js (lines 1-79 of 79)
   1  // POST /api/predict with a form field "image" (a file):
   2  // 1. send it to the model server, 2. store the image in R2, 3. store the result in D1, 4. answer the page.
   3  
   4  export async function onRequestPost({ request, env }) {
   5    const form = await request.formData();
   6    const file = form.get("image");
   7    if (!file || typeof file === "string") {
   8      return Response.json({ error: "Send an image file in the field 'image'." }, { status: 400 });
   9    }
  10    if (file.size > 5 * 1024 * 1024) {
  11      return Response.json({ error: "Image too large (max 5 MB)." }, { status: 413 });
  12    }
  13    const bytes = new Uint8Array(await file.arrayBuffer());
  14  
  15    // 1. Ask the model
  16    let result;
  17    try {
  18      result = await callModel(env, toBase64(bytes));
  19    } catch (err) {
  20      // The most common reason: the free model server was asleep and is still

Three things to notice:
- **The model call** is the same two-step call as `call_model` in section 2.5, with a **timeout**: if the server is asleep, the user gets a clear message instead of a page that hangs.
- **`env.BUCKET.put`** stores the uploaded file in R2. **`env.DB.prepare(...).bind(...)`** stores the row in D1. The `?` placeholders keep user input **out of the SQL text**: never build SQL by gluing strings, or a file named `'); DROP TABLE predictions; --` becomes a command.
- Each upload gets a **random id** (`crypto.randomUUID()`), used as the row key and the file name. Two uploads at the same moment can never overwrite each other.

## 3.5 The database (D1) and the storage (R2)

In [16]:
show_file(APP_DIR / "schema.sql")

--- cloudflare_app/schema.sql (lines 1-12 of 12)
   1  -- One row per prediction. Run with: npx wrangler d1 execute hackathon-example-db --local --file=schema.sql
   2  CREATE TABLE IF NOT EXISTS predictions (
   3    id          TEXT PRIMARY KEY,               -- a random id (crypto.randomUUID)
   4    created_at  TEXT NOT NULL,                  -- ISO date and time, UTC
   5    image_key   TEXT NOT NULL,                  -- where the uploaded image is stored in R2
   6    file_name   TEXT,                           -- the name of the uploaded file
   7    score       REAL NOT NULL,                  -- reconstruction error returned by the model server
   8    threshold   REAL NOT NULL,                  -- threshold used by the model server
   9    is_anomaly  INTEGER NOT NULL                -- 1 = anomaly, 0 = normal (SQLite has no boolean)
  10  );
  11  
  12  CREATE INDEX IF NOT EXISTS idx_predictions_created_at ON predictions (created_at);


💡 **Tip – your own table:** keep `id` and `created_at`, then add one column per value you want to keep: your inputs (for a form: one column per field) and your model's outputs. SQLite types are simple: `TEXT`, `INTEGER`, `REAL`.

## 3.6 Run it on your computer

Wrangler simulates Pages, Functions, D1 and R2 **locally**, with no Cloudflare account needed. Keep the model server of section 2.4 running, and in a second terminal:

```bash
cd cloudflare_app
npm install                                   # once
# the local secrets and settings (never commit .dev.vars):
printf 'APP_PASSWORD=choose-a-password\nMODEL_URL=http://127.0.0.1:7860\n' > .dev.vars
npm run db:local                              # create the table in the local database
npm run dev                                   # starts the app at http://localhost:8788
```

Values in `.dev.vars` replace the ones in `wrangler.toml` while you work locally: here, the local model server instead of the Render one.

Open http://localhost:8788, log in, upload an image from `model_app/examples/`. Leave the terminal open: it shows every request and every error of your functions.

The cell below tests the running app from Python: no cookie → refused; log in; predict; history; the stored image. Start `npm run dev` first; otherwise the cell tells you so.

In [17]:
APP_URL = "http://localhost:8788"
APP_PASSWORD = "choose-a-password"      # the one in your .dev.vars

try:
    requests.get(APP_URL, timeout=3)
    app_running = True
except requests.ConnectionError:
    app_running = False
    print("The app is not running: start it with `npm run dev` in cloudflare_app/, then run this cell again.")

if app_running:
    # 1. Without logging in, the API refuses
    r = requests.get(f"{APP_URL}/api/history", timeout=10)
    print("1. history without login:", r.status_code, r.json())

    # 2. Log in (the session keeps the cookie, like a browser)
    session = requests.Session()
    r = session.post(f"{APP_URL}/api/login", data={"password": APP_PASSWORD}, allow_redirects=False, timeout=10)
    print("2. login:", r.status_code, "-> cookie set:", "auth" in session.cookies)

    # 3. Predict: the Function calls the model server, stores the image in R2 and a row in D1
    with open(MODEL_APP_DIR / "examples" / "sneaker.png", "rb") as f:
        r = session.post(f"{APP_URL}/api/predict", files={"image": ("sneaker.png", f, "image/png")}, timeout=120)
    saved = r.json()
    print("3. predict:", r.status_code, {k: v for k, v in saved.items() if k != "reconstruction_png_base64"})

    # 4. The row is in the database, and the image can be read back from R2
    rows = session.get(f"{APP_URL}/api/history", timeout=10).json()
    print("4. history: the newest row is ours:", rows[0]["id"] == saved["id"], "| rows stored:", len(rows))
    image = session.get(f"{APP_URL}/api/images/{saved['id']}", timeout=10)
    same = image.content == (MODEL_APP_DIR / "examples" / "sneaker.png").read_bytes()
    print("   image from R2:", image.status_code, image.headers.get("Content-Type"), "| identical to the upload:", same)

1. history without login: 401 {'error': 'Not logged in'}
2. login: 302 -> cookie set: True
3. predict: 200 {'id': 'b575e8fb-29bf-4197-8ab0-6d72a7327cc2', 'created_at': '2026-11-16T23:02:53.182Z', 'score': 0.05946, 'threshold': 0.03506, 'is_anomaly': True, 'score_vs_threshold': 1.7, 'inverted_input': False}
4. history: the newest row is ours: True | rows stored: 1
   image from R2: 200 image/png | identical to the upload: True


The whole chain works, one step per line:
1. **No cookie, no data:** the API answers **401**. A visitor without the password sees nothing but the login page.
2. **Login** sets the cookie (a hash of the password, not the password).
3. **Predict** returns the model's answer (score 1.70× the threshold: anomaly), plus the `id` and time under which it was saved.
4. **History** reads the row back from D1, and the image read back from R2 is **byte for byte** the file we uploaded.

This is also the test to run on your own app, locally and once it is online (change `APP_URL`).

You can also look inside the local database directly:

```bash
npx wrangler d1 execute hackathon-example-db --local --command "SELECT created_at, file_name, score, is_anomaly FROM predictions"
```

## 3.7 Put it online

```bash
npx wrangler d1 create hackathon-example-db          # prints a database_id: paste it into wrangler.toml
npm run db:remote                                    # create the table in the online database
npx wrangler r2 bucket create hackathon-example-images
npx wrangler pages project create hackathon-example-app --production-branch main
npx wrangler pages secret put APP_PASSWORD           # type the password when asked
npm run deploy                                       # uploads public/ and functions/
```

Before deploying, set `MODEL_URL` in `wrangler.toml` to your **Render** address: online, Cloudflare cannot reach the server on your laptop. Wrangler prints the app's address, `https://<project>.pages.dev`. Change the names (`hackathon-example-...`) to your team's names in **both** the commands and `wrangler.toml`.

⚠️ **Watch out:** R2 must be enabled once in the Cloudflare dashboard (R2 → Overview) before `r2 bucket create` works. It is free up to 10 GB, but Cloudflare asks for a payment method. If nobody in the team can add one, leave R2 out: store only the results in D1, and remove the `BUCKET` lines from `predict.js` and `wrangler.toml`.

### Your turn (Wednesday)
1. Copy `cloudflare_app/` into your team repository. In `wrangler.toml`, set your project name, your `MODEL_URL` and your database and bucket names.
2. Change `schema.sql` and the `INSERT` in `predict.js` to store **your** inputs and outputs.
3. Change `index.html` so it shows your result in a way that makes sense for your users.
4. Run it locally (3.6), then put it online (3.7) behind your password.
5. Draw its sequence diagram (section 1 is a template).

# 4. Thursday: diagrams and failure cases

## 4.1 Three diagrams

| Diagram | Question it answers | Template |
|---|---|---|
| **Architecture** | Which parts exist and who talks to whom? | Section 1, first drawing |
| **User flow** | What does the user see and do, step by step? | Below |
| **Sequence** | What messages go between the parts, in what order? | Section 1, second drawing |

```
 USER FLOW
 ┌──────────┐  no cookie   ┌──────────┐  wrong password: try again
 │ Open app │ ───────────▶ │  Login   │ ◀──────┐
 └────┬─────┘              └────┬─────┘ ───────┘
      │ cookie OK               │ right password
      ▼                         ▼
 ┌─────────────────────────────────────┐
 │  Upload page + saved results        │
 └────────────────┬────────────────────┘
                  │ Analyse
     ┌────────────┼──────────────────┐
     ▼            ▼                  ▼
 ┌────────┐  ┌──────────────┐  ┌────────────────┐
 │ Result │  │ "Model waking │  │ "Image too     │
 │ saved  │  │  up: try      │  │  large"        │
 └───┬────┘  │  again"       │  └────────────────┘
     ▼       └──────────────┘
 ┌─────────────────┐
 │ History updated │
 └─────────────────┘
```

## 4.2 What happens when…

These are the questions of the hackathon plan, answered for the example app. Answer them for **yours**.

| Situation | What the example app does | Where |
|---|---|---|
| **The model server is asleep** | The first call waits up to 60 s, then the user sees "The model did not answer… try again in a minute". The server keeps waking up; the second try works. | `fetchWithTimeout` in `predict.js` |
| **The user reloads the page** | The current result disappears (it lived in the browser), but the **saved results** are read again from D1 and are all still there. | `loadHistory()` in `index.html` |
| **Two users save at the same time** | Each request has its own random id and its own `INSERT`; D1 handles the writes one after the other. Nothing is overwritten. | `crypto.randomUUID()` in `predict.js` |
| **Wrong password** | Back to the login page with a message; the API answers 401. | `_middleware.js`, `login.js` |
| **A huge file** | Refused with "Image too large" before anything is sent to the model. | `predict.js` |
| **The model server crashes** | Same message as a sleeping server; the error detail is in the response and in the Wrangler log. | `predict.js` |

🤔 **Think about it:** if two people **change** the same row (a counter, a status), the last write wins. Use one SQL statement that does the change (`UPDATE ... SET n = n + 1`), not "read the value, add one, write it back" in JavaScript.

## 4.3 Working with AI coding tools

An AI coding agent can write most of `predict.js`, the SQL and the page. It works best when you:
- give it **context**: this notebook, your `schema.sql`, your model server's API (inputs and outputs), and the error message you see;
- ask for **small steps** and run each one (`npm run dev`) before the next;
- **read** what it wrote: you present it on Friday and must be able to explain every part of the sequence diagram;
- **never** paste tokens or passwords into a prompt. Secrets belong in `.dev.vars` and `wrangler pages secret put`.

Agents can also be given **skills** (written instructions for a recurring task, like "deploy our app") and **connectors** (MCP servers that let them use a service, like GitHub or a database). Whatever a tool does on your behalf, check it as you would check a teammate's work.

# 5. Friday: demo checklist

| Criterion (points) | Check before 10:00 |
|---|---|
| **Working app online** (3) | The `pages.dev` address opens on a phone and on a laptop; one upload end to end works |
| **Model online, callable from another program** (2) | `wake_up` and one `call_model` from this notebook work on the Render address; README explains the autoencoder |
| **Data stored in a database + password** (2) | A reload still shows the saved results; a private window asks for the password |
| **Diagrams and three statements** (2) | Architecture, user flow and sequence diagrams; three statements ready |
| **Autoencoder homework** (1) | Setups compared and interpreted (notebook 1, section 10) |

**Wake the server** (run before the demo, with `MODEL_URL` set to your Render address):

In [18]:
if wake_up(MODEL_URL):
    start = time.time()
    answer = call_model(MODEL_URL, image_to_base64(MODEL_APP_DIR / "examples" / "coat.png"))
    print(f"ready for the demo: answered in {time.time() - start:.2f} s, score {answer['score']:.4f}")

awake: answered in 0.0 s (attempt 1)
ready for the demo: answered in 0.01 s, score 0.0124


**Three statements** (one slide):
1. **What is clear:** what works and what you understand well.
2. **What is unclear:** what you are not sure about (in the model, the data or the app).
3. **What you would do next:** with one more week.

**The 7-minute demo:** the problem and the user (1 min) → live: log in, one input, the result, reload to show it was saved (3 min) → the diagrams (2 min) → the three statements (1 min).

In [19]:
# Stop the local model server started in section 2.4
if server is not None:
    server.terminate()
    print("local model server stopped")

local model server stopped


# 6. Conclusions

### What you built this week
- A **model server**: the autoencoder of notebook 1 exported to ONNX, served by Gradio with a page for people and a `/predict` API for programs. It runs on your computer and, for free, on Render (under 200 MB of memory).
- **Calls from other programs**: `gradio_client` and two plain HTTP requests, the same ones the web app makes.
- A **serverless web app** on Cloudflare: a page, a password gate, a function that calls the model, a D1 table that keeps every result and an R2 bucket that keeps every upload.
- **Diagrams** and answers for the failure cases: a sleeping server, a reload, two saves at once.

### Limitations of the example
- The free model server sleeps after 15 minutes: the first request after a pause is slow.
- The password is shared by the whole team, and the cookie lasts 7 days. Real apps use personal accounts (for example Cloudflare Access).
- The app trusts the model's answer. A real tool would show its uncertainty and keep a human in the loop for decisions.

### References
- Gradio: [querying Gradio apps with curl](https://www.gradio.app/guides/querying-gradio-apps-with-curl) and the [Python client](https://www.gradio.app/guides/getting-started-with-the-python-client)
- ONNX Runtime: [onnxruntime.ai](https://onnxruntime.ai/docs/get-started/with-python.html); PyTorch: [exporting to ONNX](https://docs.pytorch.org/docs/stable/onnx.html)
- Render: [deploy a Python web service](https://render.com/docs/your-first-deploy) and [Blueprints](https://render.com/docs/infrastructure-as-code)
- Cloudflare: [Pages Functions](https://developers.cloudflare.com/pages/functions/), [D1](https://developers.cloudflare.com/d1/), [R2](https://developers.cloudflare.com/r2/), [Wrangler](https://developers.cloudflare.com/workers/wrangler/)